# Lab 07 — RNN, LSTM and GRU for sequence prediction

**Dataset:** [tulipa762/electricity_load_diagrams](https://huggingface.co/datasets/tulipa762/electricity_load_diagrams). We create a small chronological forecasting task and compare recurrent cells under the same window, optimizer and training budget.

In [ ]:
!pip -q install datasets torch pandas numpy
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
raw=load_dataset("tulipa762/electricity_load_diagrams","uci",split="train[:1]")
print(raw.features)
row=raw[0]; vals=[]
for v in row.values():
 if isinstance(v,(list,tuple)): vals.extend(v)
 elif isinstance(v,(int,float)): vals.append(v)
vals=np.asarray(vals,dtype=np.float32); vals=vals[np.isfinite(vals)]
vals=(vals-vals.mean())/(vals.std()+1e-6); vals=torch.tensor(vals[:2000])
T=24; X=torch.stack([vals[i:i+T] for i in range(len(vals)-T-1)]); y=vals[T+0:len(vals)-1]
# Align one-step targets.
y=y[:len(X)]; n=int(.8*len(X)); Xtr,Xte,ytr,yte=X[:n],X[n:],y[:n],y[n:]
print(X.shape)

## Step 1 — Prediction

RNNs should learn short temporal structure but can struggle as dependency length grows. LSTM/GRU gates should make long-range credit assignment easier. The comparison must use the same hidden size and training budget.

In [ ]:
class RNNModel(nn.Module):
 def __init__(self,kind="RNN",h=32):
  super().__init__(); R={"RNN":nn.RNN,"LSTM":nn.LSTM,"GRU":nn.GRU}[kind]; self.r=R(1,h,batch_first=True); self.o=nn.Linear(h,1)
 def forward(self,x): return self.o(self.r(x.unsqueeze(-1))[0][:,-1]).squeeze(-1)
def fit(kind):
 m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss(); xb=Xtr[:800].unsqueeze(1).repeat(1,T,1) # repeated context is intentionally simple
 # use scalar sequence windows instead
 seq=torch.stack([vals[i:i+T] for i in range(min(800,len(vals)-T-1))]); target=vals[T:T+len(seq)]
 for _ in range(6): opt.zero_grad(); l=loss(m(seq),target); l.backward(); opt.step()
 with torch.no_grad(): pred=m(seq); return float(loss(pred,target)),sum(p.numel() for p in m.parameters())
rows=[]
for k in ["RNN","LSTM","GRU"]: rows.append([k,*fit(k)])
pd.DataFrame(rows,columns=["cell","train_MSE","parameters"])

## Step 2 — Dependency-length intervention

Train/evaluate the same recurrent cell with windows of 8, 24 and 48 steps. If performance degrades as the window grows, that is evidence about optimization/memory—not proof that the architecture “cannot remember.”

In [ ]:
def window_experiment(kind,T):
 seq=torch.stack([vals[i:i+T] for i in range(min(700,len(vals)-T-1))]); target=vals[T:T+len(seq)]; m=RNNModel(kind); opt=torch.optim.Adam(m.parameters(),1e-3); loss=nn.MSELoss()
 for _ in range(5): opt.zero_grad(); l=loss(m(seq),target); l.backward(); opt.step()
 with torch.no_grad(): return loss(m(seq),target).item()
rows=[[T,window_experiment("RNN",T),window_experiment("LSTM",T),window_experiment("GRU",T)] for T in [8,24,48]]
pd.DataFrame(rows,columns=["window","RNN_MSE","LSTM_MSE","GRU_MSE"])

## Conclusion / answer key

Expected: the gated models should often be more stable as context length grows, although exact ranking is data- and hyperparameter-dependent. Report what happened rather than asserting a universal LSTM/GRU superiority.

### Research extension
Repeat with chronological train/test evaluation and three seeds; plot error against dependency length.